# (7065) Fredschaaf — простая астрометрическая калибровка

Этот notebook повторяет идею из `exam/astrometric_calibration.ipynb`, но работает из корня проекта и сразу показывает качество привязки координат.

Здесь всего одна задача: для одного WCS-решённого FITS-кадра сопоставить звёзды Gaia, измерить их центры и построить простую линейную модель `x, y → ξ, η`. Итог — RMS остатков в миллисекундах дуги (mas).

Это стартовая оценка, а не окончательная высокоточная редукция. Сложный конвейер для серий сохранён в `reductions_advanced.ipynb`.

In [ ]:
from pathlib import Path
import io

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from astropy import units as u
from astropy.coordinates import Distance, SkyCoord
from astropy.io import fits
from astropy.time import Time, TimeDelta
from astropy.visualization import AsinhStretch, ImageNormalize
from astropy.wcs import WCS
from photutils.centroids import centroid_2dg

plt.rcParams.update({"figure.figsize": (10, 7), "image.cmap": "magma"})

## 1. Откройте один кадр

По умолчанию используется тот же WCS-решённый тестовый кадр, что и в `exam`. Можно подставить другой FITS, но для этой первой версии в его заголовке уже должен быть WCS.

In [ ]:
ROOT = Path.cwd()
FITS_PATH = ROOT / "exam" / "Fredschaaf_R.fits"

with fits.open(FITS_PATH, memmap=False) as hdul:
    image = hdul[0].data.astype(np.float32)
    header = hdul[0].header.copy()

wcs = WCS(header)
assert wcs.has_celestial, "В выбранном кадре нет WCS — сначала нужен plate solving."

height, width = image.shape
print(f"Кадр: {FITS_PATH.name}")
print(f"Размер: {width} × {height} пикселей")
print(f"Центр WCS: RA={header['CRVAL1']:.5f}°, Dec={header['CRVAL2']:.5f}°")

In [ ]:
def display_norm(data):
    """Мягкая шкала, на которой видны и яркие, и слабые звёзды."""
    finite = data[np.isfinite(data)]
    low, high = np.percentile(finite, (5, 99.7))
    return ImageNormalize(vmin=low, vmax=high, stretch=AsinhStretch(a=0.08))

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(projection=wcs)
view = ax.imshow(image, origin="lower", norm=display_norm(image))
ax.coords.grid(color="white", linestyle=":", alpha=0.55)
ax.set_xlabel("Прямое восхождение")
ax.set_ylabel("Склонение")
ax.set_title("Исходный кадр с WCS")
plt.colorbar(view, ax=ax, label="счёт, ADU")
plt.show()

## 2. Получите опорные звёзды Gaia

Запрос выполняется к тому же локальному сервису Gaia, что и в notebook из `exam`. Собственные движения звёзд учитываются простой поправкой к эпохе середины экспозиции.

In [ ]:
start_time = Time(header["DATE-OBS"], format="isot", scale="utc")
if "DATE-END" in header:
    end_time = Time(header["DATE-END"], format="isot", scale="utc")
    mid_time = start_time + (end_time - start_time) / 2
else:
    mid_time = start_time + TimeDelta(float(header["EXPTIME"]) / 2, format="sec")

center_ra, center_dec = header["CRVAL1"], header["CRVAL2"]
MAG_MIN, MAG_MAX = 5.0, 16.5
FOV_DEG = 0.4
GAIA_URL = "http://sfa.puldb.ru:9810"

response = requests.get(
    f"{GAIA_URL}?cmd=box&ra={center_ra}&dec={center_dec}&fov={FOV_DEG}",
    timeout=30,
)
response.raise_for_status()
gaia = pd.read_csv(io.StringIO(response.text))

# Берём только звёзды с полным набором параметров собственного движения.
gaia = gaia.dropna(subset=["ra", "dec", "parallax", "pmra", "pmdec", "ref_epoch", "phot_g_mean_mag"])
gaia = gaia[(gaia["parallax"] > 0) & gaia["phot_g_mean_mag"].between(MAG_MIN, MAG_MAX)].copy()


def propagate_gaia_to_epoch(table, epoch):
    """Перенести координаты Gaia на эпоху кадра по собственным движениям."""
    catalog_coordinates = SkyCoord(
        ra=table["ra"].to_numpy(float) * u.deg,
        dec=table["dec"].to_numpy(float) * u.deg,
        distance=Distance(parallax=table["parallax"].to_numpy(float) * u.mas),
        pm_ra_cosdec=table["pmra"].to_numpy(float) * u.mas / u.yr,
        pm_dec=table["pmdec"].to_numpy(float) * u.mas / u.yr,
        obstime=Time(table["ref_epoch"].to_numpy(float), format="jyear", scale="tcb"),
        frame="icrs",
    )
    at_epoch = catalog_coordinates.apply_space_motion(new_obstime=epoch)
    table = table.copy()
    table["ra_at_epoch"] = at_epoch.ra.deg
    table["dec_at_epoch"] = at_epoch.dec.deg
    return table


gaia = propagate_gaia_to_epoch(gaia, mid_time)
print(f"Эпоха середины экспозиции: {mid_time.isot}")
print(f"Gaia-звёзд в диапазоне {MAG_MIN} < G < {MAG_MAX}: {len(gaia)}")

## 3. Сопоставьте Gaia со снимком по существующему WCS

Кружки — ожидаемые положения опорных звёзд. Если они заметно не совпадают с центрами звёзд, сначала стоит проверить WCS, а не интерпретировать будущую RMS как точность измерений.

In [ ]:
predicted_xy = wcs.all_world2pix(
    np.column_stack([gaia["ra_at_epoch"], gaia["dec_at_epoch"]]), 0,
)
MARGIN = 100
inside = (
    (predicted_xy[:, 0] > MARGIN) & (predicted_xy[:, 0] < width - MARGIN) &
    (predicted_xy[:, 1] > MARGIN) & (predicted_xy[:, 1] < height - MARGIN)
)

reference_stars = gaia.loc[inside].copy().reset_index(drop=True)
reference_stars[["x_wcs", "y_wcs"]] = predicted_xy[inside]
print(f"Опорных звёзд внутри поля: {len(reference_stars)}")

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(projection=wcs)
ax.imshow(image, origin="lower", norm=display_norm(image))
ax.scatter(reference_stars.x_wcs, reference_stars.y_wcs, s=70,
           facecolors="none", edgecolors="cyan", linewidths=0.8, label="Gaia по WCS")
ax.coords.grid(color="white", linestyle=":", alpha=0.45)
ax.set_xlabel("Прямое восхождение")
ax.set_ylabel("Склонение")
ax.legend(loc="upper right")
ax.set_title("Предсказанные положения опорных звёзд")
plt.show()

## 4. Измерьте центры звёзд и постройте линейную модель

Для каждой опорной звезды берётся маленькое окно `21 × 21` пиксель. `centroid_2dg` подбирает в нём двумерный гаусс и возвращает центр звезды. Затем обычный метод наименьших квадратов строит линейную привязку к касательной плоскости.

In [ ]:
HALF_BOX = 10


def centroid_star(data, x, y, half_box=10):
    """Центр звезды в небольшом окне; возвращает NaN, если окно не подходит."""
    x0, y0 = int(round(x)), int(round(y))
    cutout = data[y0-half_box:y0+half_box+1, x0-half_box:x0+half_box+1]
    if cutout.shape != (2 * half_box + 1, 2 * half_box + 1):
        return np.nan, np.nan
    try:
        cx, cy = centroid_2dg(cutout - np.nanmedian(cutout))
    except Exception:
        return np.nan, np.nan
    if not (np.isfinite(cx) and np.isfinite(cy) and 0 < cx < 2 * half_box and 0 < cy < 2 * half_box):
        return np.nan, np.nan
    return x0 - half_box + cx, y0 - half_box + cy


measured = []
for star in reference_stars.itertuples():
    x, y = centroid_star(image, star.x_wcs, star.y_wcs, HALF_BOX)
    if np.isfinite(x) and np.isfinite(y):
        measured.append((x, y, star.ra_at_epoch, star.dec_at_epoch, star.phot_g_mean_mag))

measured = pd.DataFrame(measured, columns=["x", "y", "ra", "dec", "gmag"])
print(f"Успешно измерены центры: {len(measured)}")
measured.head()

In [ ]:
def tangent_plane(ra_deg, dec_deg, ra0_deg, dec0_deg):
    """Координаты ξ, η на касательной плоскости в градусах."""
    ra, dec = np.deg2rad(ra_deg), np.deg2rad(dec_deg)
    ra0, dec0 = np.deg2rad(ra0_deg), np.deg2rad(dec0_deg)
    denominator = np.sin(dec) * np.sin(dec0) + np.cos(dec) * np.cos(dec0) * np.cos(ra - ra0)
    xi = np.cos(dec) * np.sin(ra - ra0) / denominator
    eta = (np.sin(dec) * np.cos(dec0) - np.cos(dec) * np.sin(dec0) * np.cos(ra - ra0)) / denominator
    return np.rad2deg(xi), np.rad2deg(eta)


assert len(measured) >= 6, "Слишком мало центров для калибровки. Проверьте WCS и диапазон звёздных величин."
measured["xi_deg"], measured["eta_deg"] = tangent_plane(
    measured.ra.to_numpy(), measured.dec.to_numpy(), center_ra, center_dec
)

design = np.column_stack([np.ones(len(measured)), measured.x, measured.y])
coef_xi, *_ = np.linalg.lstsq(design, measured.xi_deg, rcond=None)
coef_eta, *_ = np.linalg.lstsq(design, measured.eta_deg, rcond=None)

xi_residual_deg = measured.xi_deg - design @ coef_xi
eta_residual_deg = measured.eta_deg - design @ coef_eta
measured["xi_residual_mas"] = xi_residual_deg * 3_600_000
measured["eta_residual_mas"] = eta_residual_deg * 3_600_000

# В каждой координате подогнаны три параметра: свободный член, x и y.
degrees_of_freedom = len(measured) - 3
assert degrees_of_freedom > 0, "Недостаточно опорных звёзд для оценки точности."
sigma_xi_mas = np.sqrt(np.sum(measured.xi_residual_mas ** 2) / degrees_of_freedom)
sigma_eta_mas = np.sqrt(np.sum(measured.eta_residual_mas ** 2) / degrees_of_freedom)
sigma_radial_mas = np.hypot(sigma_xi_mas, sigma_eta_mas)

print(f"ξ = {coef_xi[1]:.8g} · x + {coef_xi[2]:.8g} · y + {coef_xi[0]:.8g} deg")
print(f"η = {coef_eta[1]:.8g} · x + {coef_eta[2]:.8g} · y + {coef_eta[0]:.8g} deg")
print()
print(f"Опорных звёзд: {len(measured)}; степеней свободы: {degrees_of_freedom}")
print(f"СКО ξ: {sigma_xi_mas:.1f} mas")
print(f"СКО η: {sigma_eta_mas:.1f} mas")
print(f"Двумерное СКО: {sigma_radial_mas:.1f} mas")

## 5. Посмотрите остатки

Каждая точка — одна опорная звезда. Хорошо, когда облако компактно и не показывает выраженного наклона, дуги или зависимости от положения в кадре. Такие структуры означают, что простой линейной модели уже недостаточно.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, column, label in zip(
    axes,
    ["xi_residual_mas", "eta_residual_mas"],
    ["Остатки ξ, mas", "Остатки η, mas"],
):
    scatter = ax.scatter(measured.x, measured.y, c=measured[column], s=55, cmap="coolwarm")
    ax.axhline(0, color="none")
    ax.set_xlabel("x, пиксели")
    ax.set_ylabel("y, пиксели")
    ax.set_title(label)
    plt.colorbar(scatter, ax=ax, label="mas")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
ax.imshow(image, origin="lower", norm=display_norm(image))
ax.scatter(measured.x, measured.y, s=90, facecolors="none", edgecolors="cyan", linewidths=0.8)
for star in measured.itertuples():
    ax.text(star.x + 12, star.y + 12, f"G={star.gmag:.1f}", color="white", fontsize=7)
ax.set_xlabel("x, пиксели")
ax.set_ylabel("y, пиксели")
ax.set_title(f"Звёзды линейной калибровки; двумерное СКО = {sigma_radial_mas:.1f} mas")
plt.show()

## Как интерпретировать результат

Напечатанные СКО вычисляются по формулам `σξ = √(Σvξ² / (n − 3))` и `ση = √(Σvη² / (n − 3))`: в каждой линейной модели подогнаны свободный член и коэффициенты при `x`, `y`. `n` — число опорных звёзд. Двумерное СКО равно `√(σξ² + ση²)`.

Это разброс опорных звёзд относительно **этой простой линейной модели**. Он включает ошибки центров, исходного WCS, каталога, атмосферные искажения и неописанную дисторсию камеры. Перенос Gaia учитывает собственные движения и расстояние из параллакса, но не топоцентрическую параллактическую поправку и не DCR.

Поэтому это хороший первый контроль порядка точности, но ещё не ошибка координат астероида и не окончательный астрометрический результат.